In [2]:
# ============================================================
# Part B: Hyperparameter Optimization
# Dataset: clv_mobile_data_train.txt (continuous Y)
#
# Required packages:
#   pip install pandas numpy scikit-learn
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ------------------------------------------------------------
# 1. Load the data
# ------------------------------------------------------------
# 你的文件完整路径
path = "/Users/tangjuanzhe/Desktop/MATH 60600数据挖掘课件/donnees/clv_mobile_data_train.txt"

# 文件表头有 17 个列名，但每行数据开头还有一个未命名的 id 列。
# 因此我们跳过原表头，手动指定 18 个列名。
col_names = [
    "id",
    "numsessions", "numdays", "numpurchases", "totpurchases",
    "totplaytime", "totscore", "numlives", "numelements",
    "difflevel", "skill1", "skill2", "country", "opsys", "acquis",
    "trendsession", "trendpurchase", "y"
]

df = pd.read_csv(
    path,
    sep=r"\s+",
    engine="python",
    header=None,
    skiprows=1,
    names=col_names
)

print("Data shape:", df.shape)
print("\nFirst rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)

# ------------------------------------------------------------
# 2. Features and target
# ------------------------------------------------------------
X = df.drop(columns=["id", "y"]).copy()
y = df["y"].copy()

# 类别变量
cat_cols = ["country", "opsys", "acquis"]
num_cols = [c for c in X.columns if c not in cat_cols]

# 为了 OneHotEncoder 稳定处理，把类别列转成字符串
for col in cat_cols:
    X[col] = X[col].astype(str)

print("\nCategorical columns:", cat_cols)
print("Numerical columns:", num_cols)

# ------------------------------------------------------------
# 3. Train / validation split
# ------------------------------------------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("\nX_train:", X_train.shape)
print("X_valid:", X_valid.shape)

# ------------------------------------------------------------
# 4. Preprocessing + Random Forest regression pipeline
# ------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

pipe = Pipeline(steps=[
    ("pre", preprocessor),
    ("rf", rf)
])

# ------------------------------------------------------------
# 5. Grid Search hyperparameter optimization
# ------------------------------------------------------------
# 这里选择 Grid Search 作为 Part B 的一种超参数优化方法。
# 评价指标使用负 RMSE（因为 GridSearchCV 默认最大化分数）。
param_grid = {
    "rf__n_estimators": [100, 200],
    "rf__max_depth": [5, 10, None],
    "rf__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("\nBest parameters:")
print(grid.best_params_)

print("Best CV RMSE:", -grid.best_score_)

# ------------------------------------------------------------
# 6. Validation evaluation
# ------------------------------------------------------------
y_pred = grid.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
mae = mean_absolute_error(y_valid, y_pred)
r2 = r2_score(y_valid, y_pred)

print("\nValidation results:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

# ------------------------------------------------------------
# 7. Show top 5 parameter combinations
# ------------------------------------------------------------
results = pd.DataFrame(grid.cv_results_)
results = results.sort_values(by="rank_test_score")

print("\nTop 5 CV results:")
print(
    results[
        ["params", "mean_test_score", "rank_test_score"]
    ].head()
)

Data shape: (5000, 18)

First rows:
    id  numsessions  numdays  numpurchases  totpurchases  totplaytime  \
0  "1"            3        5             1      2.000000   202.168295   
1  "2"           12        7             3      2.000000   413.911718   
2  "3"            2        4             3     13.110147   130.995568   
3  "4"            6        6             3     21.709514   207.446863   
4  "5"            6        7             4      7.212596   214.583872   

     totscore  numlives  numelements  difflevel    skill1    skill2 country  \
0   94.783740        24           14          2  0.369258  0.553046     "6"   
1  104.587668        28           21          2  0.603684  0.379317     "2"   
2   98.178186        21           18          2  0.438464  0.445509     "1"   
3  100.547096        37           20          2  0.489058  0.519593     "1"   
4  107.447260        33           23          2  0.534398  0.675010     "2"   

   opsys  acquis  trendsession  trendpurchase     

In [3]:
# ============================================================
# Part B - Bayesian Optimization with Optuna (TPE)
# Dataset: clv_mobile_data_train.txt (continuous Y)
#
# Required packages:
#   pip install pandas numpy scikit-learn optuna
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.base import clone

import optuna

# ------------------------------------------------------------
# 1. Load data (same as Grid Search)
# ------------------------------------------------------------
path = "/Users/tangjuanzhe/Desktop/MATH 60600数据挖掘课件/donnees/clv_mobile_data_train.txt"

col_names = [
    "id",
    "numsessions", "numdays", "numpurchases", "totpurchases",
    "totplaytime", "totscore", "numlives", "numelements",
    "difflevel", "skill1", "skill2", "country", "opsys", "acquis",
    "trendsession", "trendpurchase", "y"
]

df = pd.read_csv(
    path, sep=r"\s+", engine="python",
    header=None, skiprows=1, names=col_names
)

print("Data shape:", df.shape)

X = df.drop(columns=["id", "y"]).copy()
y = df["y"].copy()

cat_cols = ["country", "opsys", "acquis"]
num_cols = [c for c in X.columns if c not in cat_cols]

for col in cat_cols:
    X[col] = X[col].astype(str)

# ------------------------------------------------------------
# 2. Train / validation split (same random_state as Grid Search)
# ------------------------------------------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

# ------------------------------------------------------------
# 3. Preprocessing
# ------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols)
    ]
)

# ------------------------------------------------------------
# 4. Objective function for Optuna
# ------------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.3, 1.0)
    }

    model = Pipeline(steps=[
        ("pre", clone(preprocessor)),
        ("rf", RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        ))
    ])

    cv = KFold(n_splits=3, shuffle=True, random_state=42)

    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    # optuna minimizes, so we return RMSE
    return -scores.mean()

# ------------------------------------------------------------
# 5. Create study with TPE sampler and optimize
# ------------------------------------------------------------
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=30, show_progress_bar=True)

print("\n=== Bayesian Optimization (Optuna TPE) ===")
print("Best trial RMSE:", study.best_value)
print("Best params:", study.best_params)
print("Number of finished trials:", len(study.trials))

# ------------------------------------------------------------
# 6. Train final model with best params and evaluate
# ------------------------------------------------------------
best_rf = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_pipe = Pipeline(steps=[
    ("pre", clone(preprocessor)),
    ("rf", best_rf)
])

best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_valid)

rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
mae = mean_absolute_error(y_valid, y_pred)
r2 = r2_score(y_valid, y_pred)

print("\nValidation results (Bayesian optimized RF):")
print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")
print(f"R2  : {r2:.4f}")

# ------------------------------------------------------------
# 7. (Optional) Top 5 best trials
# ------------------------------------------------------------
trials_df = study.trials_dataframe().sort_values("value")
print("\nTop 5 trials by RMSE:")
print(trials_df[["value", "params_n_estimators", "params_max_depth",
                 "params_min_samples_split", "params_min_samples_leaf",
                 "params_max_features"]].head())

Data shape: (5000, 18)
X_train: (4000, 16)
X_valid: (1000, 16)


  0%|          | 0/30 [00:00<?, ?it/s]


=== Bayesian Optimization (Optuna TPE) ===
Best trial RMSE: 6.0425714593921915
Best params: {'n_estimators': 250, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 0.7664888239930253}
Number of finished trials: 30

Validation results (Bayesian optimized RF):
RMSE: 5.5408
MAE : 4.2648
R2  : 0.7965

Top 5 trials by RMSE:
       value  params_n_estimators  params_max_depth  params_min_samples_split  \
27  6.042571                  250                19                         8   
29  6.045001                  200                17                         7   
28  6.068329                  250                19                         8   
26  6.082734                  250                16                         8   
23  6.089844                  150                19                         8   

    params_min_samples_leaf  params_max_features  
27                        1             0.766489  
29                        2             0.725900  
28      

In [ ]:
# ============================================================
# Partie B - Keras Tuner (RandomSearch) pour un MLP de régression
# Jeu de données : clv_mobile_data_train.txt (Y continu)
#
# Bibliothèques requises :
#   pip install pandas numpy scikit-learn tensorflow keras-tuner
#
# Remarque :
#   Le noyau a planté avec Hyperband car trop de configurations
#   et d'époques. On utilise donc ici RandomSearch, avec un
#   espace de recherche réduit, et on limite les ressources
#   de TensorFlow pour éviter le crash.
# ============================================================

# --- Limiter les ressources TensorFlow AVANT d'importer TF ---
import os

# Limiter le nombre de threads CPU utilisés par TensorFlow
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["TF_NUM_INTRAOP_THREADS"] = "2"
os.environ["TF_NUM_INTEROP_THREADS"] = "2"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # réduire les logs

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt

# --- Reproductibilité ---
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Limiter la mémoire GPU si disponible ---
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except Exception as e:
        print("GPU memory growth non configuré :", e)

# ------------------------------------------------------------
# 1. Chargement des données
# ------------------------------------------------------------
path = "/Users/tangjuanzhe/Desktop/MATH 60600数据挖掘课件/donnees/clv_mobile_data_train.txt"

# Le fichier a 17 noms de colonnes mais chaque ligne contient
# un identifiant "id" en plus. On fournit donc 18 noms.
col_names = [
    "id",
    "numsessions", "numdays", "numpurchases", "totpurchases",
    "totplaytime", "totscore", "numlives", "numelements",
    "difflevel", "skill1", "skill2", "country", "opsys", "acquis",
    "trendsession", "trendpurchase", "y"
]

df = pd.read_csv(
    path, sep=r"\s+", engine="python",
    header=None, skiprows=1, names=col_names
)

print("Dimensions des données :", df.shape)

# --- Variables explicatives et cible ---
X = df.drop(columns=["id", "y"]).copy()
y = df["y"].copy()

# --- Variables catégorielles et numériques ---
cat_cols = ["country", "opsys", "acquis"]
num_cols = [c for c in X.columns if c not in cat_cols]

# On convertit les variables catégorielles en chaînes
# pour OneHotEncoder
for col in cat_cols:
    X[col] = X[col].astype(str)

# ------------------------------------------------------------
# 2. Découpage entraînement / validation
# ------------------------------------------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print("X_train :", X_train.shape)
print("X_valid :", X_valid.shape)

# ------------------------------------------------------------
# 3. Prétraitement pour le MLP
#    - variables catégorielles : One-Hot
#    - variables numériques : StandardScaler
#    On ajuste UNIQUEMENT sur l'ensemble d'entraînement.
# ------------------------------------------------------------
pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", StandardScaler(), num_cols)
    ],
    sparse_threshold=0
)

pre.fit(X_train)

X_train_mlp = pre.transform(X_train).astype("float32")
X_valid_mlp = pre.transform(X_valid).astype("float32")

y_train_mlp = y_train.values.astype("float32")
y_valid_mlp = y_valid.values.astype("float32")

input_dim = X_train_mlp.shape[1]
print("Dimension d'entrée du MLP :", input_dim)

# ------------------------------------------------------------
# 4. Fonction de construction du modèle pour Keras Tuner
#    Espace de recherche volontairement réduit pour éviter
#    le plantage du noyau.
# ------------------------------------------------------------
def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    # Nombre de couches cachées : 1 ou 2
    n_layers = hp.Int("n_layers", min_value=1, max_value=2)

    for i in range(n_layers):
        units = hp.Choice(f"units_{i}", values=[32, 64, 128])
        dropout = hp.Choice(f"dropout_{i}", values=[0.0, 0.2, 0.3])

        model.add(layers.Dense(units, activation="relu"))
        model.add(layers.Dropout(dropout))

    # Couche de sortie : 1 neurone, activation linéaire
    model.add(layers.Dense(1))

    # Taux d'apprentissage (échantillonnage logarithmique)
    learning_rate = hp.Float(
        "learning_rate",
        min_value=1e-4,
        max_value=1e-2,
        sampling="log"
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.RootMeanSquaredError(name="rmse")]
    )
    return model

# ------------------------------------------------------------
# 5. RandomSearch avec Keras Tuner
#    - max_trials = 10 (raisonnable)
#    - executions_per_trial = 1
#    - objectif : minimiser val_rmse
# ------------------------------------------------------------
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective("val_rmse", direction="min"),
    max_trials=10,
    executions_per_trial=1,
    directory="keras_tuner_partb",
    project_name="mlp_regression_random",
    overwrite=True,
    seed=SEED
)

# --- EarlyStopping pour arrêter les essais non prometteurs ---
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_rmse",
    patience=5,
    restore_best_weights=True
)

# --- Lancement de la recherche ---
tuner.search(
    X_train_mlp, y_train_mlp,
    validation_data=(X_valid_mlp, y_valid_mlp),
    epochs=30,             # réduit pour éviter le crash mémoire
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# --- Meilleurs hyperparamètres ---
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n=== Keras Tuner (RandomSearch) ===")
print("Meilleurs hyperparamètres :")
for k, v in best_hp.values.items():
    print(f"  {k} : {v}")

# ------------------------------------------------------------
# 6. Entraînement du meilleur modèle et évaluation
# ------------------------------------------------------------
best_model = tuner.hypermodel.build(best_hp)

history = best_model.fit(
    X_train_mlp, y_train_mlp,
    validation_data=(X_valid_mlp, y_valid_mlp),
    epochs=60,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# --- Prédictions et métriques ---
y_pred_mlp = best_model.predict(X_valid_mlp, verbose=0).ravel()

rmse_mlp = np.sqrt(mean_squared_error(y_valid, y_pred_mlp))
mae_mlp = mean_absolute_error(y_valid, y_pred_mlp)
r2_mlp = r2_score(y_valid, y_pred_mlp)

print("\nRésultats sur l'ensemble de validation (MLP Keras Tuner) :")
print(f"RMSE : {rmse_mlp:.4f}")
print(f"MAE  : {mae_mlp:.4f}")
print(f"R²   : {r2_mlp:.4f}")